# 🧠 Augmentasi Data Deteksi Emosi Bahasa Indonesia

**Model:** `meta-llama/Meta-Llama-3-8B-Instruct`  
**Metode:** Instruction-based paraphrasing (label-preserving)  
**Target:** 1500 sample per label emosi  
**Label:** `joy`, `sad`, `anger`, `fear`, `love`, `neutral`

---

### 📋 Alur Pipeline
1. Install & import library
2. Load dataset CSV
3. Tampilkan distribusi label awal
4. Identifikasi kelas yang perlu diaugmentasi
5. Hitung jumlah augmentasi yang dibutuhkan tiap label
6. Buat prompt template augmentasi
7. Generate data baru menggunakan Llama-3-8B-Instruct
8. Filter duplicate output
9. Gabungkan dataset asli dan hasil augmentasi
10. Tampilkan distribusi label akhir
11. Simpan hasil akhir ke CSV baru

> ⚠️ **Pastikan runtime Google Colab diset ke GPU A100** sebelum menjalankan notebook ini.

---
## ⚙️ Langkah 1 — Install Library

In [ ]:
# Install semua dependensi yang diperlukan
# transformers : load model Llama-3
# bitsandbytes : quantisasi 4-bit agar model muat di VRAM
# accelerate   : distribusi model ke GPU secara otomatis
# pandas, tqdm : manipulasi data dan progress bar

!pip install -q transformers
!pip install -q bitsandbytes
!pip install -q accelerate
!pip install -q pandas tqdm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 43.9 MB/s eta 0:00:00


---
## 📦 Langkah 2 — Import Library & Konfigurasi Global

In [ ]:
import os
import random
import warnings
import pandas as pd
import numpy as np
import torch
from tqdm import tqdm
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
)

warnings.filterwarnings("ignore")

# --- Seed global agar hasil reproducible di setiap run ---
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("✅ Library berhasil diimpor")

✅ Library berhasil diimpor


In [ ]:
# ============================================================
# KONFIGURASI UTAMA
# Sesuaikan bagian ini sebelum menjalankan pipeline
# ============================================================


CONFIG = {
    # Path file CSV dataset train yang sudah di-split
    "train_csv_path": "train.csv",

    # Path output CSV hasil augmentasi
    "output_csv_path": "train_augmented.csv",

    # Nama model Llama 3 yang digunakan untuk augmentasi
    "model_name": "meta-llama/Meta-Llama-3-8B-Instruct",

    # Target jumlah sample per label emosi
    "target_per_label": 1500,

    # Label emosi yang digunakan dalam dataset
    "emotion_labels": ["joy", "sad", "anger", "fear", "love", "neutral"],

    # Jumlah teks yang di-generate dalam satu batch (sesuaikan dengan VRAM)
    "batch_size": 32,

    # Maksimal percobaan ulang jika output tidak valid
    "max_retries": 3,

    # Parameter generation model
    "generation_config": {
        "temperature": 0.7,   # Kontrol kreativitas (0=deterministik, 1=acak)
        "top_p": 0.9,         # Nucleus sampling: ambil token dari top 90% probabilitas
        "do_sample": True,    # Aktifkan sampling (wajib jika temperature/top_p digunakan)
        "max_new_tokens": 64, # Maksimal token output baru per generasi
    },
}

print("✅ Konfigurasi berhasil dimuat")
print(f"   Model  : {CONFIG['model_name']}")
print(f"   Target : {CONFIG['target_per_label']} sample/label")
print(f"   Labels : {CONFIG['emotion_labels']}")

✅ Konfigurasi berhasil dimuat
   Model  : meta-llama/Meta-Llama-3-8B-Instruct
   Target : 1500 sample/label
   Labels : ['joy', 'sad', 'anger', 'fear', 'love', 'neutral']


---
## 📂 Langkah 3 — Load Dataset CSV

In [ ]:
def load_dataset(csv_path: str) -> pd.DataFrame:
    """
    Memuat dataset dari file CSV.
    Memastikan kolom yang diperlukan tersedia (index, tweet, label).
    """
    print(f"\n📂 Memuat dataset dari: {csv_path}")
    df = pd.read_csv(csv_path)

    # Validasi kolom yang diperlukan
    required_cols = {"tweet", "label"}
    missing = required_cols - set(df.columns)
    if missing:
        raise ValueError(f"Kolom berikut tidak ditemukan di CSV: {missing}")

    # Hapus baris dengan nilai kosong pada kolom kunci
    df = df.dropna(subset=["tweet", "label"])

    # Pastikan kolom tweet bertipe string
    df["tweet"] = df["tweet"].astype(str).str.strip()

    # Normalisasi label ke huruf kecil
    df["label"] = df["label"].str.lower().str.strip()

    # Buat kolom index jika belum ada
    if "index" not in df.columns:
        df = df.reset_index(drop=False)  # jadikan index DataFrame sebagai kolom

    print(f"✅ Dataset berhasil dimuat: {len(df)} baris")
    print(f"   Kolom : {list(df.columns)}")
    return df


# Jalankan fungsi load
df_train = load_dataset(CONFIG["train_csv_path"])
df_train.head()


📂 Memuat dataset dari: train.csv
✅ Dataset berhasil dimuat: 4956 baris
   Kolom : ['index', 'tweet', 'label']


,index,tweet,label
0,6933,"gapapa masih ada besok, ayoo tetap menyerah ja...",sad
1,2985,wkwk sama bangga banget merasa bener nebak jeo...,joy
2,6412,sedih aa. tahan ngantuk kot,sad
3,1791,tolong panik ini melanda lagi,fear
4,6490,rindumu sajak sentuhan. kalimatku pernah pucat...,sad


---
## 📊 Langkah 4 & 5 — Distribusi Label Awal & Kebutuhan Augmentasi

In [ ]:
def analyze_distribution(df: pd.DataFrame, title: str = "Distribusi Label") -> dict:
    """
    Menampilkan distribusi jumlah sample per label emosi
    dan mengembalikan dictionary {label: jumlah}.
    """
    print(f"\n📊 {title}")
    print("-" * 40)
    dist = df["label"].value_counts().to_dict()

    for label in CONFIG["emotion_labels"]:
        count = dist.get(label, 0)
        bar = "█" * (count // 20)  # visualisasi bar sederhana
        print(f"  {label:<10}: {count:>5}  {bar}")

    print(f"\n  Total: {len(df)} baris")
    return dist


def identify_augmentation_needs(dist: dict) -> dict:
    """
    Mengidentifikasi label mana yang perlu diaugmentasi
    dan berapa jumlah sample tambahan yang dibutuhkan.

    Hanya augmentasi label dengan jumlah < target_per_label.
    Label yang sudah >= target tidak diaugmentasi.
    """
    target = CONFIG["target_per_label"]
    needs = {}

    print(f"\n🎯 Target per label: {target} sample")
    print("\n📋 Kebutuhan augmentasi:")
    print("-" * 50)

    for label in CONFIG["emotion_labels"]:
        current = dist.get(label, 0)
        needed = max(0, target - current)
        needs[label] = needed
        status = "✅ Cukup" if needed == 0 else f"⚠️  Butuh +{needed} sample"
        print(f"  {label:<10}: {current:>5} saat ini → {status}")

    return needs


# Tampilkan distribusi awal
dist_awal = analyze_distribution(df_train, title="Distribusi Label AWAL")

# Identifikasi kebutuhan augmentasi
augmentation_needs = identify_augmentation_needs(dist_awal)


📊 Distribusi Label AWAL
----------------------------------------
  joy       :   890  ████████████████████████████████████████████
  sad       :   706  ███████████████████████████████████
  anger     :   774  ██████████████████████████████████████
  fear      :   630  ███████████████████████████████
  love      :   544  ███████████████████████████
  neutral   :  1412  ██████████████████████████████████████████████████████████████████████

  Total: 4956 baris

🎯 Target per label: 1500 sample

📋 Kebutuhan augmentasi:
--------------------------------------------------
  joy       :   890 saat ini → ⚠️  Butuh +610 sample
  sad       :   706 saat ini → ⚠️  Butuh +794 sample
  anger     :   774 saat ini → ⚠️  Butuh +726 sample
  fear      :   630 saat ini → ⚠️  Butuh +870 sample
  love      :   544 saat ini → ⚠️  Butuh +956 sample
  neutral   :  1412 saat ini → ⚠️  Butuh +88 sample


---
## 🤖 Langkah 6 — Load Model Llama-3-8B-Instruct (4-bit Quantization)

In [ ]:
# Cek ketersediaan dan spesifikasi GPU
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    bf16_support = torch.cuda.is_bf16_supported()
    print(f"🖥️  GPU terdeteksi : {gpu_name}")
    print(f"   VRAM          : {vram:.1f} GB")
    print(f"   BFloat16      : {'✅ Didukung (optimal untuk A100)' if bf16_support else '❌ Tidak didukung'}")
else:
    print("⚠️  GPU tidak terdeteksi. Proses akan berjalan di CPU (sangat lambat).")
    print("   → Pastikan runtime Colab diset ke A100 GPU")

🖥️  GPU terdeteksi : NVIDIA A100-SXM4-40GB
   VRAM          : 42.4 GB
   BFloat16      : ✅ Didukung (optimal untuk A100)


In [ ]:
def load_llama_model():
    """
    Memuat model Llama-3-8B-Instruct dengan konfigurasi:
    - Quantisasi 4-bit (NF4) menggunakan bitsandbytes → hemat VRAM ~4x
    - BFloat16 untuk komputasi (optimal di A100)
    - Accelerate untuk distribusi memori otomatis (device_map='auto')
    """
    model_name = CONFIG["model_name"]
    print(f"\n🤖 Memuat model: {model_name}")
    print("⏳ Proses ini membutuhkan beberapa menit pertama kali (download + load)...")

    # Konfigurasi quantisasi 4-bit untuk efisiensi memori
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,                      # Aktifkan quantisasi 4-bit
        bnb_4bit_quant_type="nf4",              # NormalFloat4: akurasi lebih baik dari int4
        bnb_4bit_compute_dtype=torch.bfloat16,  # BF16 untuk forward pass (A100 optimal)
        bnb_4bit_use_double_quant=True,         # Double quantisasi: hemat ~0.4 bit/param
    )

    # Load tokenizer (kamus kata → token ID)
    tokenizer = AutoTokenizer.from_pretrained(
        model_name,
        padding_side="left",  # Padding di kiri penting untuk batch generation (causal LM)
        trust_remote_code=True,
    )

    # Tambahkan pad token jika belum ada (Llama-3 default tidak punya pad token)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
        tokenizer.pad_token_id = tokenizer.eos_token_id

    # Load model dengan quantisasi 4-bit
    # Catatan: jangan tambahkan .to() setelah ini — model 4-bit bitsandbytes
    # sudah otomatis ditempatkan ke device yang benar oleh device_map='auto'
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        quantization_config=bnb_config,
        device_map="auto",           # Distribusi otomatis ke GPU/CPU
        trust_remote_code=True,
    )

    # CATATAN: jangan panggil model.eval() atau model.to() pada model 4-bit bitsandbytes
    # karena keduanya memicu ValueError. Gunakan torch.inference_mode() saat generate.

    # Deteksi device utama model via hf_device_map (lebih aman untuk model multi-device)
    if hasattr(model, 'hf_device_map') and model.hf_device_map:
        # Ambil device dari layer pertama yang terdaftar
        first_device = list(model.hf_device_map.values())[0]
        model_device = torch.device(f"cuda:{first_device}" if isinstance(first_device, int) else first_device)
    else:
        model_device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    print("\n✅ Model berhasil dimuat!")
    print(f"   Device    : {model_device}")
    return tokenizer, model, model_device


# Load model dan tokenizer
# model_device digunakan untuk memindahkan input tensor ke GPU secara eksplisit
tokenizer, model, model_device = load_llama_model()


🤖 Memuat model: meta-llama/Meta-Llama-3-8B-Instruct
⏳ Proses ini membutuhkan beberapa menit pertama kali (download + load)...


config.json:   0%|          | 0.00/654 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/73.0 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/187 [00:00<?, ?B/s]


✅ Model berhasil dimuat!
   Device    : cuda


---
## 📝 Langkah 7 — Prompt Template Augmentasi

In [ ]:
# Deskripsi emosi dalam Bahasa Indonesia untuk memperkaya konteks prompt
# Ini membantu model memahami nuansa emosi yang harus dipertahankan
EMOTION_DESCRIPTIONS = {
    "joy":     "kebahagiaan, kesenangan, kegembiraan, atau antusiasme",
    "sad":     "kesedihan, dukacita, kekecewaan, atau rasa kehilangan",
    "anger":   "kemarahan, frustrasi, kejengkelan, atau ketidakpuasan",
    "fear":    "ketakutan, kekhawatiran, kecemasan, atau rasa was-was",
    "love":    "cinta, kasih sayang, rasa suka, atau rasa peduli",
    "neutral": "pernyataan netral tanpa emosi yang menonjol",
}


def build_augmentation_prompt(original_text: str, label: str) -> str:
    """
    Membuat prompt instruksi untuk Llama-3 agar melakukan
    parafrase teks sambil mempertahankan emosi dan makna aslinya.

    Menggunakan format ChatML resmi Llama-3-Instruct:
      <|begin_of_text|>
      <|start_header_id|>system<|end_header_id|>\n\n{system}<|eot_id|>
      <|start_header_id|>user<|end_header_id|>\n\n{user}<|eot_id|>
      <|start_header_id|>assistant<|end_header_id|>\n\n
    """
    emotion_desc = EMOTION_DESCRIPTIONS.get(label, label)

    system_msg = (
        "Kamu adalah asisten parafrase teks berbahasa Indonesia. "
        "Tugasmu adalah menulis ulang teks dengan kata-kata yang berbeda "
        "namun tetap mempertahankan emosi dan makna utama teks asli. "
        "Hasilkan HANYA teks parafrase saja, tanpa penjelasan tambahan."
    )

    user_msg = (
        f"Tulis ulang tweet berikut dengan kata-kata yang berbeda.\n"
        f"Pertahankan emosi: {label} ({emotion_desc}).\n"
        f"Gunakan bahasa Indonesia yang natural dan hindari kalimat yang terlalu mirip aslinya.\n\n"
        f"Tweet asli: {original_text}\n\n"
        f"Hasil parafrase:"
    )

    # Format ChatML untuk Llama-3-Instruct
    prompt = (
        f"<|begin_of_text|>"
        f"<|start_header_id|>system<|end_header_id|>\n\n{system_msg}<|eot_id|>"
        f"<|start_header_id|>user<|end_header_id|>\n\n{user_msg}<|eot_id|>"
        f"<|start_header_id|>assistant<|end_header_id|>\n\n"
    )

    return prompt


# Contoh prompt untuk verifikasi
contoh_prompt = build_augmentation_prompt("Hari ini sangat menyenangkan!", "joy")
print("📝 Contoh prompt yang akan dikirim ke model:")
print("-" * 60)
print(contoh_prompt)

📝 Contoh prompt yang akan dikirim ke model:
------------------------------------------------------------
<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Kamu adalah asisten parafrase teks berbahasa Indonesia. Tugasmu adalah menulis ulang teks dengan kata-kata yang berbeda namun tetap mempertahankan emosi dan makna utama teks asli. Hasilkan HANYA teks parafrase saja, tanpa penjelasan tambahan.<|eot_id|><|start_header_id|>user<|end_header_id|>

Tulis ulang tweet berikut dengan kata-kata yang berbeda.
Pertahankan emosi: joy (kebahagiaan, kesenangan, kegembiraan, atau antusiasme).
Gunakan bahasa Indonesia yang natural dan hindari kalimat yang terlalu mirip aslinya.

Tweet asli: Hari ini sangat menyenangkan!

Hasil parafrase:<|eot_id|><|start_header_id|>assistant<|end_header_id|>




---
## 🔍 Langkah 8 — Fungsi Validasi & Filter Duplikat

In [ ]:
def is_valid_output(generated: str, original: str, existing_texts: set) -> bool:
    """
    Memvalidasi apakah hasil generasi memenuhi kriteria kualitas:

    1. Panjang minimum  : setidaknya 5 kata
    2. Anti-duplikat    : tidak ada di existing_texts (exact match)
    3. Anti-plagiat     : Jaccard similarity < 0.85 terhadap teks asli
    4. Bebas artefak    : tidak mengandung token/marker dari prompt
    """
    gen = generated.strip()

    # Validasi 1: Panjang minimum
    if len(gen.split()) < 5:
        return False

    # Validasi 2: Tidak duplikat (exact match)
    if gen in existing_texts:
        return False

    # Validasi 3: Jaccard similarity < 0.85 terhadap teks asli
    # Jaccard = |irisan token| / |gabungan token|
    orig_tokens = set(original.lower().split())
    gen_tokens  = set(gen.lower().split())
    if orig_tokens and gen_tokens:
        intersection = orig_tokens & gen_tokens
        union = orig_tokens | gen_tokens
        jaccard = len(intersection) / len(union)
        if jaccard > 0.85:
            return False  # Terlalu mirip dengan asli

    # Validasi 4: Bebas dari artefak token prompt Llama
    artefak = ["tweet asli:", "hasil parafrase:", "<|", "|>",
               "system", "user", "assistant"]
    gen_lower = gen.lower()
    for artefak_str in artefak:
        if artefak_str in gen_lower:
            return False

    return True  # Semua validasi lolos


def extract_generated_text(full_output: str, prompt: str) -> str:
    """
    Mengekstrak teks hasil generasi dari output model
    dengan membuang bagian prompt yang ikut muncul di output.
    """
    # Strategi 1: Hapus bagian prompt jika ada di output
    if prompt in full_output:
        generated = full_output[len(prompt):]
    else:
        # Strategi 2: Ambil bagian setelah marker asisten terakhir
        marker = "<|start_header_id|>assistant<|end_header_id|>\n\n"
        if marker in full_output:
            generated = full_output.split(marker)[-1]
        else:
            generated = full_output

    # Bersihkan token khusus Llama dari output
    for tok in ["<|eot_id|>", "<|end_of_text|>", "<|begin_of_text|>"]:
        generated = generated.replace(tok, "")

    # Ambil hanya baris pertama (1 tweet = 1 kalimat)
    generated = generated.strip().split("\n")[0].strip()

    return generated


print("✅ Fungsi validasi dan ekstraksi teks berhasil didefinisikan")

✅ Fungsi validasi dan ekstraksi teks berhasil didefinisikan


---
## ⚡ Langkah 9 — Generate Data Augmentasi (Batching)

In [ ]:
def generate_augmented_samples(
    source_texts: list,
    label: str,
    num_needed: int,
    tokenizer,
    model,
    model_device,
    existing_texts: set,
) -> list:
    """
    Menghasilkan data augmentasi untuk satu label emosi menggunakan batch generation.

    Strategi:
    - Pilih teks sumber secara acak dari kelas yang sama
    - Generate batch_size sample sekaligus untuk efisiensi GPU
    - Validasi setiap output sebelum diterima
    - Loop sampai jumlah target terpenuhi

    Args:
        source_texts   : daftar teks asli sebagai sumber parafrase
        label          : label emosi yang sedang diaugmentasi
        num_needed     : jumlah sample baru yang dibutuhkan
        tokenizer      : tokenizer Llama
        model          : model Llama yang sudah dimuat
        existing_texts : set teks yang sudah ada (untuk cek duplikat)

    Returns:
        List teks baru hasil augmentasi
    """
    batch_size = CONFIG["batch_size"]
    gen_config  = CONFIG["generation_config"]
    max_retries = CONFIG["max_retries"]

    augmented  = []                  # Kumpulkan hasil yang valid
    seen_texts = set(existing_texts) # Salinan lokal untuk cek duplikat

    print(f"\n  Mengaugmentasi '{label}': membutuhkan {num_needed} sample baru")

    with tqdm(total=num_needed, desc=f"  [{label}]", unit="sample") as pbar:

        while len(augmented) < num_needed:
            # Hitung batch size yang dibutuhkan (dengan buffer retry)
            remaining     = num_needed - len(augmented)
            current_batch = min(batch_size, remaining * max_retries)

            # Pilih teks sumber secara acak (dengan penggantian)
            selected_sources = random.choices(source_texts, k=current_batch)

            # Buat prompt untuk setiap teks sumber dalam batch
            prompts = [
                build_augmentation_prompt(src, label)
                for src in selected_sources
            ]

            # Tokenisasi seluruh batch sekaligus
            inputs = tokenizer(
                prompts,
                return_tensors="pt",
                padding=True,       # Padding agar panjang seragam dalam batch
                truncation=True,    # Potong jika melebihi max_length
                max_length=512,
            )
            # Pindahkan setiap tensor input ke device GPU secara eksplisit
            # (.to() pada model 4-bit bitsandbytes tidak didukung, jadi kita
            #  pindahkan tensor input-nya saja, bukan modelnya)
            inputs = {k: v.to(model_device) for k, v in inputs.items()}

            # Generate output (inference_mode lebih aman dari no_grad untuk model 4-bit)
            with torch.inference_mode():
                outputs = model.generate(
                    **inputs,
                    temperature=gen_config["temperature"],
                    top_p=gen_config["top_p"],
                    do_sample=gen_config["do_sample"],
                    max_new_tokens=gen_config["max_new_tokens"],
                    pad_token_id=tokenizer.pad_token_id,
                    eos_token_id=tokenizer.eos_token_id,
                )

            # Proses setiap output dalam batch
            for i, output_ids in enumerate(outputs):
                if len(augmented) >= num_needed:
                    break

                # Decode token ID → teks (dengan token khusus untuk ekstraksi)
                full_text = tokenizer.decode(output_ids, skip_special_tokens=False)
                generated = extract_generated_text(full_text, prompts[i])

                # Terima output jika lolos semua validasi
                if is_valid_output(generated, selected_sources[i], seen_texts):
                    augmented.append(generated)
                    seen_texts.add(generated)  # Daftarkan agar tidak duplikat
                    pbar.update(1)

            # Bersihkan cache GPU setelah setiap batch
            torch.cuda.empty_cache()

    print(f"  ✅ Berhasil menghasilkan {len(augmented)} sample baru untuk '{label}'")
    return augmented


print("✅ Fungsi generate_augmented_samples berhasil didefinisikan")

✅ Fungsi generate_augmented_samples berhasil didefinisikan


In [ ]:
# Siapkan set teks yang sudah ada untuk cek duplikat global
existing_texts = set(df_train["tweet"].tolist())

# Tentukan index awal untuk data augmentasi
# Gunakan max index + 1 agar tidak bentrok dengan index data asli
max_existing_index = df_train["index"].max() if "index" in df_train.columns else len(df_train)
next_index = int(max_existing_index) + 1

# Jalankan generasi untuk setiap label yang membutuhkan augmentasi
all_augmented_rows = []

for label, num_needed in augmentation_needs.items():

    # Skip label yang sudah cukup
    if num_needed == 0:
        print(f"\n  ⏭️  '{label}' sudah cukup ({dist_awal.get(label, 0)} sample). Dilewati.")
        continue

    # Ambil semua teks asli untuk label ini sebagai sumber parafrase
    source_texts = df_train[df_train["label"] == label]["tweet"].tolist()

    if not source_texts:
        print(f"\n  ⚠️  Tidak ada teks sumber untuk label '{label}'. Dilewati.")
        continue

    # Generate sample baru
    new_texts = generate_augmented_samples(
        source_texts=source_texts,
        label=label,
        num_needed=num_needed,
        tokenizer=tokenizer,
        model=model,
        model_device=model_device,
        existing_texts=existing_texts,
    )

    # Simpan hasil dengan index unik yang tidak bentrok
    for text in new_texts:
        all_augmented_rows.append({
            "index": next_index,
            "tweet": text,
            "label": label,
        })
        next_index += 1

    # Tambahkan teks baru ke set global
    existing_texts.update(new_texts)

print(f"\n🎉 Total sample augmentasi berhasil dibuat: {len(all_augmented_rows)}")


  Mengaugmentasi 'joy': membutuhkan 610 sample baru


  [joy]: 100%|██████████| 610/610 [24:01<00:00,  2.36s/sample]


  ✅ Berhasil menghasilkan 610 sample baru untuk 'joy'

  Mengaugmentasi 'sad': membutuhkan 794 sample baru


  [sad]: 100%|██████████| 794/794 [42:28<00:00,  3.21s/sample]


  ✅ Berhasil menghasilkan 794 sample baru untuk 'sad'

  Mengaugmentasi 'anger': membutuhkan 726 sample baru


...

  ✅ Berhasil menghasilkan 870 sample baru untuk 'fear'

  Mengaugmentasi 'love': membutuhkan 956 sample baru


  [love]: 100%|██████████| 956/956 [24:35<00:00,  1.54s/sample]


  ✅ Berhasil menghasilkan 956 sample baru untuk 'love'

  Mengaugmentasi 'neutral': membutuhkan 88 sample baru


  [neutral]: 100%|██████████| 88/88 [02:46<00:00,  1.90s/sample]

  ✅ Berhasil menghasilkan 88 sample baru untuk 'neutral'

🎉 Total sample augmentasi berhasil dibuat: 4044


---
## 🔀 Langkah 10 — Gabungkan Dataset & Tampilkan Distribusi Akhir

In [ ]:
# Buat DataFrame dari hasil augmentasi
df_augmented = pd.DataFrame(all_augmented_rows)

print(f"📦 Dataset asli      : {len(df_train):>6} baris")
print(f"📦 Data augmentasi   : {len(df_augmented):>6} baris")

# Gabungkan dataset asli dengan hasil augmentasi
df_final = pd.concat([df_train, df_augmented], ignore_index=True)

# Pastikan kolom sesuai format output yang diinginkan: index, tweet, label
df_final = df_final[["index", "tweet", "label"]].copy()
df_final["tweet"] = df_final["tweet"].astype(str).str.strip()
df_final["label"] = df_final["label"].str.lower().str.strip()

print(f"\n📦 Dataset gabungan  : {len(df_final):>6} baris")

# Tampilkan distribusi label akhir
dist_akhir = analyze_distribution(df_final, title="Distribusi Label AKHIR (setelah augmentasi)")

📦 Dataset asli      :   4956 baris
📦 Data augmentasi   :   4044 baris

📦 Dataset gabungan  :   9000 baris

📊 Distribusi Label AKHIR (setelah augmentasi)
----------------------------------------
  joy       :  1500  ███████████████████████████████████████████████████████████████████████████
  sad       :  1500  ███████████████████████████████████████████████████████████████████████████
  anger     :  1500  ███████████████████████████████████████████████████████████████████████████
  fear      :  1500  ███████████████████████████████████████████████████████████████████████████
  love      :  1500  ███████████████████████████████████████████████████████████████████████████
  neutral   :  1500  ███████████████████████████████████████████████████████████████████████████

  Total: 9000 baris


In [ ]:
# Tampilkan ringkasan perubahan per label
print("\n📈 Ringkasan Augmentasi per Label:")
print("-" * 55)
print(f"  {'Label':<10}  {'Sebelum':>8}  {'Sesudah':>8}  {'Tambahan':>10}")
print("-" * 55)

for label in CONFIG["emotion_labels"]:
    before = dist_awal.get(label, 0)
    after  = df_final[df_final["label"] == label].shape[0]
    added  = after - before
    print(f"  {label:<10}  {before:>8}  {after:>8}  {'+' + str(added):>10}")

print("-" * 55)
print(f"  {'TOTAL':<10}  {len(df_train):>8}  {len(df_final):>8}  {'+' + str(len(df_augmented)):>10}")

# Tampilkan beberapa contoh hasil augmentasi
print("\n📋 Contoh hasil augmentasi:")
df_augmented.sample(min(5, len(df_augmented)), random_state=SEED)


📈 Ringkasan Augmentasi per Label:
-------------------------------------------------------
  Label        Sebelum   Sesudah    Tambahan
-------------------------------------------------------
  joy              890      1500        +610
  sad              706      1500        +794
  anger            774      1500        +726
  fear             630      1500        +870
  love             544      1500        +956
  neutral         1412      1500         +88
-------------------------------------------------------
  TOTAL           4956      9000       +4044

📋 Contoh hasil augmentasi:


,index,tweet,label
3633,10713,"""Cinta pertama ketika lihat wajahnya, tapi ter...",love
149,7229,Gembira dengan kejutan yang tidak diharapkan!,joy
2025,9105,"Apa pun yang dipost di medsos, justru menunjuk...",anger
2505,9585,pas oper oper pucknya tuh bisa ngepas gitu man...,fear
3203,10283,"lengket, packagingnya terlihat mewah & pemakai...",love


---
## 💾 Langkah 11 — Simpan Dataset Hasil Augmentasi ke CSV

In [ ]:
# Simpan dataset final ke CSV
output_path = CONFIG["output_csv_path"]
df_final.to_csv(output_path, index=False)

print(f"💾 Dataset hasil augmentasi disimpan ke: {output_path}")
print(f"   Jumlah baris : {len(df_final)}")
print(f"   Kolom        : {list(df_final.columns)}")

# Verifikasi file berhasil disimpan
df_verify = pd.read_csv(output_path)
print(f"\n✅ Verifikasi file berhasil: {len(df_verify)} baris terbaca")
df_verify.head()

💾 Dataset hasil augmentasi disimpan ke: train_augmented.csv
   Jumlah baris : 9000
   Kolom        : ['index', 'tweet', 'label']

✅ Verifikasi file berhasil: 9000 baris terbaca


,index,tweet,label
0,6933,"gapapa masih ada besok, ayoo tetap menyerah ja...",sad
1,2985,wkwk sama bangga banget merasa bener nebak jeo...,joy
2,6412,sedih aa. tahan ngantuk kot,sad
3,1791,tolong panik ini melanda lagi,fear
4,6490,rindumu sajak sentuhan. kalimatku pernah pucat...,sad


In [ ]:
# (Opsional) Download file CSV langsung dari Google Colab ke komputer lokal
try:
    from google.colab import files
    files.download(output_path)
    print(f"📥 File '{output_path}' sedang diunduh ke komputer lokal...")
except ImportError:
    print("ℹ️  Bukan di Google Colab. Unduh file secara manual dari file manager.")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

📥 File 'train_augmented.csv' sedang diunduh ke komputer lokal...


---
## 🎉 Pipeline Selesai!

Dataset hasil augmentasi telah disimpan dengan format:

| Kolom | Deskripsi |
|-------|-----------|
| `index` | ID unik setiap baris (tidak ada duplikat dengan data asli) |
| `tweet` | Teks tweet (asli atau hasil augmentasi) |
| `label` | Label emosi: `joy`, `sad`, `anger`, `fear`, `love`, `neutral` |

### ✅ Checklist Kualitas
- [x] Hanya kelas dengan jumlah < 1500 yang diaugmentasi
- [x] Tidak ada duplikat exact-match
- [x] Jaccard similarity < 0.85 terhadap teks sumber
- [x] Index baru tidak bentrok dengan index data asli
- [x] Emosi dan makna utama teks dipertahankan
- [x] Output natural dalam Bahasa Indonesia